# Sesion 09 - ML distribuido: regresion con Spark MLlib

En la sesion anterior usamos datos de observabilidad del pipeline Kafka + Spark para construir una primera regresion de latencia.

En esta sesion formalizaremos ese flujo como un pipeline distribuido de Spark MLlib.

Logro de la sesion:

```text
Ahora puedo construir un pipeline ML distribuido, entrenarlo, evaluarlo y reutilizarlo.
```

Usaremos Spark MLlib, no Pandas ni scikit-learn.

## 1. Idea general

Queremos predecir la latencia promedio del siguiente minuto usando metricas del minuto actual.

Ejemplo conceptual:

```text
metricas minuto actual -> latencia promedio del siguiente minuto
```

En Spark ML el flujo se organiza con tres ideas clave:

- `Transformer`: recibe un DataFrame y devuelve otro DataFrame transformado.
- `Estimator`: aprende desde los datos y produce un modelo.
- `Pipeline`: une varios pasos para que el entrenamiento sea reproducible.

En esta practica el pipeline sera:

```text
VectorAssembler -> StandardScaler -> LinearRegression
```

## 2. Crear SparkSession

Creamos una sesion local de Spark. Aunque estemos en una maquina, el codigo usa APIs distribuidas de Spark.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("s09-ml-distribuido-regresion-mllib") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

spark

## 3. Leer datos del pipeline

Primero intentamos leer los datos Parquet generados en las practicas de streaming y observabilidad.

Si esos archivos no existen en el entorno, se genera un dataset sintetico para que la clase pueda continuar.

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

candidate_paths = [
    "../artifacts/output/orden_eventos_observabilidad_p1",
    "../artifacts/output/orden_eventos_parquet",
]

df = None
source_path = None

for path in candidate_paths:
    if Path(path).exists():
        try:
            df = spark.read.parquet(path)
            source_path = path
            break
        except Exception as exc:
            print(f"No se pudo leer {path}: {exc}")

if df is None:
    print("No se encontraron Parquet reales. Se creara un dataset sintetico para la practica.")
    base = spark.range(0, 240)
    df = base.select(
        (F.timestamp_seconds(F.lit(1746300000) + F.col("id") * 20)).alias("kafkaTimestamp"),
        (
            F.lit(120)
            + (F.col("id") % 12) * 8
            + F.when((F.col("id") % 37) == 0, 180).otherwise(0)
            + (F.rand(seed=42) * 30)
        ).cast("long").alias("latencyMs"),
        F.lit("ordenes").alias("topic"),
        (F.col("id") % 3).cast("int").alias("partition"),
        F.col("id").cast("long").alias("offset"),
        F.lit("demo").alias("origen"),
        F.lit(True).alias("isValid")
    )
    source_path = "dataset sintetico"

print(f"Fuente de datos: {source_path}")
df.printSchema()
df.show(5, truncate=False)

## 4. Construir serie temporal de latencia

El modelo no usara eventos individuales. Primero agregamos por ventanas de 1 minuto.

Cada fila representa el comportamiento del pipeline durante un minuto.

In [ ]:
def build_latency_series(input_df):
    return input_df \
        .filter(F.col("kafkaTimestamp").isNotNull()) \
        .filter(F.col("latencyMs").isNotNull()) \
        .groupBy(F.window("kafkaTimestamp", "1 minute")) \
        .agg(
            F.count("*").alias("eventCount"),
            F.avg("latencyMs").alias("avgLatencyMs"),
            F.min("latencyMs").alias("minLatencyMs"),
            F.max("latencyMs").alias("maxLatencyMs"),
            F.stddev("latencyMs").alias("stdLatencyMs")
        ) \
        .withColumn("windowStart", F.col("window.start")) \
        .withColumn("windowEnd", F.col("window.end")) \
        .drop("window") \
        .na.fill({"stdLatencyMs": 0.0}) \
        .orderBy("windowStart")

serie_latency = build_latency_series(df)
window_count = serie_latency.count()

if window_count < 20:
    print(f"Solo hay {window_count} ventanas. Se usara una serie sintetica mas amplia para entrenar sin problemas.")
    synthetic_base = spark.range(0, 360)
    df = synthetic_base.select(
        (F.timestamp_seconds(F.lit(1746300000) + F.col("id") * 20)).alias("kafkaTimestamp"),
        (
            F.lit(120)
            + (F.col("id") % 12) * 8
            + F.when((F.col("id") % 41) == 0, 190).otherwise(0)
            + (F.rand(seed=99) * 35)
        ).cast("long").alias("latencyMs")
    )
    serie_latency = build_latency_series(df)
    window_count = serie_latency.count()

serie_latency.show(20, truncate=False)
print("Cantidad de ventanas:", window_count)

## 5. Preparar features y label

La columna `label` sera la latencia promedio del siguiente minuto.

Para construirla usamos `lead`, una funcion de ventana analitica. Asi convertimos la serie temporal en un problema supervisado.

In [ ]:
from pyspark.sql.window import Window

w = Window.orderBy("windowStart")

dataset_ml = serie_latency \
    .withColumn("label", F.lead("avgLatencyMs", 1).over(w)) \
    .withColumn("latencyRangeMs", F.col("maxLatencyMs") - F.col("minLatencyMs")) \
    .withColumn("hourOfDay", F.hour("windowStart")) \
    .withColumn("minuteOfHour", F.minute("windowStart")) \
    .select(
        "windowStart",
        "eventCount",
        "avgLatencyMs",
        "minLatencyMs",
        "maxLatencyMs",
        "stdLatencyMs",
        "latencyRangeMs",
        "hourOfDay",
        "minuteOfHour",
        "label"
    ) \
    .na.drop()

dataset_ml.show(20, truncate=False)
print("Filas para ML:", dataset_ml.count())

## 6. Separar entrenamiento y prueba

Como estamos trabajando con una serie temporal, usaremos una separacion cronologica: las primeras filas entrenan y las ultimas prueban.

Esto evita entrenar con informacion del futuro.

In [ ]:
ordered = dataset_ml.withColumn("rowNumber", F.row_number().over(w))
total_rows = ordered.count()
train_cutoff = int(total_rows * 0.8)

train = ordered.filter(F.col("rowNumber") <= train_cutoff).drop("rowNumber")
test = ordered.filter(F.col("rowNumber") > train_cutoff).drop("rowNumber")

print("Filas train:", train.count())
print("Filas test:", test.count())

train.select("windowStart", "avgLatencyMs", "label").show(5, truncate=False)
test.select("windowStart", "avgLatencyMs", "label").show(5, truncate=False)

## 7. Crear Pipeline de Spark ML

Spark ML espera que las variables predictoras esten dentro de una columna vectorial llamada normalmente `features`.

En este pipeline:

- `VectorAssembler` une columnas numericas en un vector.
- `StandardScaler` escala las variables para que tengan magnitudes comparables.
- `LinearRegression` entrena el modelo de regresion distribuido.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression

feature_cols = [
    "eventCount",
    "avgLatencyMs",
    "minLatencyMs",
    "maxLatencyMs",
    "stdLatencyMs",
    "latencyRangeMs",
    "hourOfDay",
    "minuteOfHour",
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="rawFeatures",
    handleInvalid="keep"
)

scaler = StandardScaler(
    inputCol="rawFeatures",
    outputCol="features",
    withStd=True,
    withMean=True
)

lr = LinearRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    maxIter=50,
    regParam=0.05,
    elasticNetParam=0.0
)

pipeline = Pipeline(stages=[assembler, scaler, lr])

pipeline

## 8. Entrenar el modelo distribuido

Cuando ejecutamos `fit`, Spark distribuye las transformaciones y el entrenamiento sobre las particiones del DataFrame.

In [ ]:
pipeline_model = pipeline.fit(train)

predictions = pipeline_model.transform(test)

predictions.select(
    "windowStart",
    "avgLatencyMs",
    "label",
    "prediction"
).show(20, truncate=False)

## 9. Evaluar el modelo

Usaremos tres metricas:

- `RMSE`: penaliza mas los errores grandes.
- `MAE`: error promedio absoluto, facil de interpretar.
- `R2`: proporcion de variabilidad explicada por el modelo.

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

metrics = {}

for metric_name in ["rmse", "mae", "r2"]:
    evaluator = RegressionEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric_name
    )
    metrics[metric_name] = evaluator.evaluate(predictions)

for name, value in metrics.items():
    print(f"{name.upper()}: {value:.4f}")

## 10. Interpretar resultados

Ahora revisamos la diferencia entre el valor real y la prediccion.

Esto ayuda a pasar de la metrica numerica a una lectura operacional: cuantos milisegundos se esta equivocando el modelo.

In [ ]:
error_df = predictions \
    .withColumn("errorMs", F.col("label") - F.col("prediction")) \
    .withColumn("absErrorMs", F.abs(F.col("errorMs")))

error_df.select(
    "windowStart",
    F.round("label", 2).alias("realNextAvgLatencyMs"),
    F.round("prediction", 2).alias("predictedNextAvgLatencyMs"),
    F.round("errorMs", 2).alias("errorMs"),
    F.round("absErrorMs", 2).alias("absErrorMs")
).orderBy(F.desc("absErrorMs")).show(20, truncate=False)

## 11. Ver parametros aprendidos

Como el ultimo paso del pipeline es una regresion lineal, podemos inspeccionar sus coeficientes.

Ojo: como usamos `StandardScaler`, los coeficientes corresponden a variables escaladas.

In [ ]:
lr_model = pipeline_model.stages[-1]

print("Intercept:", lr_model.intercept)

coef_rows = list(zip(feature_cols, lr_model.coefficients.toArray().tolist()))
coef_df = spark.createDataFrame(coef_rows, ["feature", "coefficient"])

coef_df.orderBy(F.desc(F.abs(F.col("coefficient")))).show(truncate=False)

## 12. Guardar y cargar el modelo

Un modelo de ML no termina al entrenarse. Debe poder guardarse y reutilizarse.

Guardaremos el `PipelineModel`, no solo la regresion. Asi se conservan tambien los pasos de preparacion de features.

In [ ]:
from pyspark.ml import PipelineModel

model_path = "../artifacts/models/ml_latency_pipeline_lr"

pipeline_model.write().overwrite().save(model_path)

loaded_model = PipelineModel.load(model_path)

loaded_predictions = loaded_model.transform(test)
loaded_predictions.select("windowStart", "label", "prediction").show(5, truncate=False)

print(f"Modelo guardado en: {model_path}")

## 13. Reutilizar el modelo sobre nuevos datos batch

Para simular datos nuevos, tomaremos las ultimas ventanas de la serie.

La sesion 10 usara esta misma idea, pero aplicandola sobre datos nuevos y luego sobre streaming.

In [ ]:
new_batch = test.drop("label")

# Para evaluar en esta practica volvemos a unir la etiqueta real.
# En produccion normalmente la etiqueta futura aun no existe.
new_batch_for_demo = test

batch_predictions = loaded_model.transform(new_batch_for_demo)

batch_predictions.select(
    "windowStart",
    "avgLatencyMs",
    "prediction"
).show(10, truncate=False)

## 14. Que observar en Spark UI

Durante la ejecucion, revisar:

- Jobs creados por `fit` y `transform`.
- Stages generados por agregaciones y joins internos.
- Particiones usadas por el DataFrame.
- Tiempo de ejecucion del entrenamiento.

La idea no es solo ver el resultado del modelo, sino entender que el entrenamiento tambien es un trabajo distribuido.

## 15. Cierre

En esta sesion construimos un pipeline ML distribuido completo:

```text
datos Parquet -> serie temporal -> features -> Pipeline ML -> evaluacion -> modelo guardado
```

Lo importante:

- MLlib trabaja con DataFrames distribuidos.
- `Pipeline` permite reproducir el flujo completo.
- El modelo guardado incluye transformaciones y algoritmo.
- La evaluacion debe interpretarse en unidades del problema, aqui milisegundos de latencia.

Proxima sesion:

```text
Aplicar el modelo entrenado sobre datos nuevos y sobre streaming.
```

## Ejercicios sugeridos

1. Cambiar el tamano de ventana de 1 minuto a 30 segundos o 2 minutos.
2. Agregar una variable `previousAvgLatencyMs` usando `lag`.
3. Comparar el split cronologico con `randomSplit`.
4. Modificar `regParam` y observar cambios en RMSE, MAE y R2.
5. Guardar las predicciones en Parquet para analizarlas en otra sesion.